# Appendix A2: Hierarchical Bayesian models, a deeper look

Companion notebook. The figures are produced by `generate.py`; this notebook walks one variant of the hierarchical model so you can change the data and refit cell by cell.

Reference: [chapter.md](chapter.md). The model used here is the same logit-link hierarchical structure introduced in Chapter 11 Loop D and refined in Appendix A1's funnel discussion.

In [ ]:
import arviz as az
import numpy as np
import pymc as pm
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
az.style.use('arviz-darkgrid')

## A small synthetic A/B with mixed segment sizes

In [ ]:
rng = np.random.default_rng(101)
K = 8
true_effects = rng.normal(0.05, 0.04, size=K)
sizes = [800, 800, 800, 800, 80, 80, 80, 80]
base_p = 0.30
data = []
for eff, n in zip(true_effects, sizes):
    s_c = rng.binomial(n, base_p)
    s_t = rng.binomial(n, np.clip(base_p + eff, 1e-4, 1 - 1e-4))
    data.append((int(s_c), n, int(s_t), n))
for i, (sc, nc, st, nt) in enumerate(data):
    print(f'seg{i} (n={nc:4d})  control {sc:4d}/{nc} = {sc/nc:.3f}   treatment {st:4d}/{nt} = {st/nt:.3f}   diff = {st/nt - sc/nc:+.3f}   true = {true_effects[i]:+.3f}')

## Hierarchical fit (non-centered, prior tau ~ HalfNormal(0.5))

In [ ]:
s_c = np.array([d[0] for d in data]); n_c = np.array([d[1] for d in data])
s_t = np.array([d[2] for d in data]); n_t = np.array([d[3] for d in data])
with pm.Model():
    baseline = pm.Normal('baseline', mu=0, sigma=2, shape=K)
    mu = pm.Normal('mu', mu=0, sigma=1)
    tau = pm.HalfNormal('tau', sigma=0.5)
    z = pm.Normal('z', mu=0, sigma=1, shape=K)
    effect = pm.Deterministic('effect', mu + tau * z)
    p_c = pm.Deterministic('p_c', pm.math.sigmoid(baseline))
    p_t = pm.Deterministic('p_t', pm.math.sigmoid(baseline + effect))
    pm.Binomial('obs_c', n=n_c, p=p_c, observed=s_c)
    pm.Binomial('obs_t', n=n_t, p=p_t, observed=s_t)
    idata = pm.sample(1500, tune=1500, chains=2, random_seed=2002, progressbar=False, target_accept=0.95)
az.summary(idata, var_names=['mu', 'tau'])

## Posterior on the per-segment lift, vs no-pooling estimates

In [ ]:
p_t = idata.posterior['p_t'].values
p_c = idata.posterior['p_c'].values
diffs = (p_t - p_c).reshape(-1, K)
for i in range(K):
    indep = data[i][2] / data[i][3] - data[i][0] / data[i][1]
    lo, hi = np.quantile(diffs[:, i], 0.025), np.quantile(diffs[:, i], 0.975)
    print(f'seg{i}  no-pooling = {indep:+.3f}   partial-pooling 95% CI = [{lo:+.3f}, {hi:+.3f}]   true = {true_effects[i]:+.3f}')